#### ENRICH `SELLERS` FROM BRONZE TO SILVER

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'SELLERS_ENRICHED'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
SOURCE = f"{CATALOG}.BRONZE.SELLERS"
AUDIT_TABLE = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"

In [0]:
sellers_df = spark.table(SOURCE)
sellers_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

sellers_enriched_df = sellers_df.select(
    "seller_id",
    "seller_zip_code_prefix",
    upper(trim(col("seller_city"))).alias("seller_city"),
    upper(trim(col("seller_state"))).alias("seller_state"),
    "file_name",
    "file_path",
    current_timestamp().alias('load_timestamp')
).filter('seller_id IS NOT NULL')

display(sellers_enriched_df.limit(1))

In [0]:
sellers_enriched_df.createOrReplaceTempView('enriched_sellers_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.SELLERS_ENRICHED`

In [0]:
from pyspark.sql.functions import *

def write_audit(run_id, status, start_time, rows=None, error=None):
  end = datetime.now()
  row = (run_id, "SELLERS_ENRICHED_LOAD", "SILVER", SOURCE, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
  (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None


try:
  result = spark.sql(f'''MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY seller_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_sellers_temp_vw
  )
  WHERE rn = 1
) src
ON (src.seller_id = tgt.seller_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.seller_id = src.seller_id,
    tgt.seller_zip_code_prefix = src.seller_zip_code_prefix,
    tgt.seller_city = src.seller_city,
    tgt.seller_state = src.seller_state
WHEN NOT MATCHED THEN
INSERT (seller_id, seller_zip_code_prefix, seller_city, seller_state, origin_file_path, load_timestamp)
VALUES (src.seller_id, src.seller_zip_code_prefix, src.seller_city, src.seller_state, src.file_path, current_timestamp)
''')
  
  m = result.first().asDict()
  inserted = m.get("num_inserted_rows") or 0
  updated  = m.get("num_updated_rows") or 0
  rows_written = int(inserted + updated)
  print(f"inserted={inserted}, updated={updated}")

except Exception:
  status, err = "FAILED", traceback.format_exc()
  raise
finally:
  write_audit(run_id, status, start_time, rows_written, err)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.SELLERS_ENRICHED')